# Setup

In [165]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [166]:
# to load the virtual environment
## & "$HOME\nlp_owlapy_env_py311\Scripts\Activate.ps1"

In [167]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0


In [168]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


torch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU 0: NVIDIA RTX A2000 8GB Laptop GPU
Using DEVICE = cuda:0


# Imports

In [169]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc

from dataclasses import dataclass, asdict
from itertools import product

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [170]:
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer

In [171]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import annotate_doc_with_ocean_folder
from ocean.ocean_schema import (
    require_ocean_layer,
    register_spacy_ocean_extension
)

In [172]:
from ontology_graph_builder import build_networkx_graph_from_ttl, ontology_label_tree

from cluster_typing.ontology_probability_scoring import (
    OntologyTraversalConfig,
    OntologyScoringConfig,
    OntologyMentionWeightConfig,
    OntologyEvidenceExportConfig,
    export_ontology_evidence_jsonls,
)

from cluster_typing.ontology_annotator import (
    OntologyAnnotationConfig,
    annotate_doc_with_ontology_folder,
)

from cluster_typing.ontology_schema import (
    register_spacy_ontology_extension,
    require_ontology_layer,
)


In [173]:
from ontology.ontology_managment import (
    assert_cluster_relation,
    assert_cluster_type,
    assert_cluster_value,
    load_tbox,
    save_ontology,
)

# Functions

In [174]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [175]:
import re

def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Remove standalone page numbers, e.g.
    # This removes only lines that contain digits plus optional whitespace.
    text = re.sub(r"(?m)^[ \t]*\d+[ \t]*$", "", text)

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [176]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str,
    local_path: str | Path,
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [177]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    register_spacy_ontology_extension()
    register_spacy_ocean_extension()
    
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [178]:
# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"./resources/books/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / f"outputs_[{TEXT_FILENAME}]"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_ROOT = PROJECT_DIR / "resources/ontologies"

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
COREF_DOC_PATH = OUTPUT_ROOT / "coref_doc.pkl"
ONTOLOGY_TYPED_DOC_PATH = OUTPUT_ROOT / "ontology_typed_doc.pkl"
OCEAN_SCORED_DOC_PATH = OUTPUT_ROOT / "ocean_scored_doc.pkl"

ONTOLOGY_TTL_PATH = ONTOLOGY_ROOT/ "narrative_entity_ontology_clean.ttl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("COREF_DOC_PATH =", COREF_DOC_PATH)
print("ONTOLOGY_TYPED_DOC_PATH =", ONTOLOGY_TYPED_DOC_PATH)
print("OCEAN_SCORED_DOC_PATH =", OCEAN_SCORED_DOC_PATH)
print("ONTOLOGY_TTL_PATH =", ONTOLOGY_TTL_PATH)


TEXT_PATH = resources\books\oz_full.txt
OUTPUT_ROOT = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]
TOKENIZED_DOC_PATH = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\tokenized_doc.pkl
COREF_DOC_PATH = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\coref_doc.pkl
ONTOLOGY_TYPED_DOC_PATH = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ontology_typed_doc.pkl
OCEAN_SCORED_DOC_PATH = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ocean_scored_doc.pkl
ONTOLOGY_TTL_PATH = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\resources\ontologies\narrative_entity_ontology_clean.ttl


In [179]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\global_coref


## Runtime and pipeline config

In [180]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


RUNTIME_PROFILE = {'kind': 'local_cuda', 'env': 'Local', 'cuda_available': True, 'gpu_count': 1, 'gpu_name': 'NVIDIA RTX A2000 8GB Laptop GPU', 'device': 'cuda:0', 'cpu_load_first': True, 'precision_policy': 'auto', 'p100_fallback_to_float32': False, 'default_chunk_size': 6000, 'default_overlap_sentences': 60}
DEVICE = cuda:0
CHUNK_SIZE(expanded) = 6000
OVERLAP_SENTENCES = 60
MAX_EXPANDED_CHUNK_TOKENS = 6000
GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\global_coref


# Main

### Tokenization

In [181]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

[doc] Loading tokenized Doc from c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\tokenized_doc.pkl
Doc tokens: 48045
Doc sentences: 1943


### Chunking

In [182]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Ontology building


In [183]:
onto, graph = load_tbox(ONTOLOGY_TTL_PATH, require_property_descriptions=True)

print(graph.number_of_nodes(), "classes")
print(graph.number_of_edges(), "subclass edges")

25 classes
22 subclass edges


## Node extraction

### Coreference clusters extraction

In [184]:
if COREF_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {COREF_DOC_PATH}")
    doc = load_doc(COREF_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, COREF_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", COREF_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

[coref-annotation] Loading coref-annotated Doc from c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\coref_doc.pkl


In [185]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

[coref-annotation] Summary:
{'n_mentions': 9460, 'n_clusters': 566, 'n_indexed_tokens': 11462, 'n_indexed_heads': 7346, 'n_indexed_spans': 7375}
Non-singleton clusters: 199

cluster_id=8 | canonical_name='Dorothy' | n_mentions=1900

cluster_id=97 | canonical_name='the Scarecrow' | n_mentions=1032

cluster_id=153 | canonical_name='the Lion' | n_mentions=837

cluster_id=48 | canonical_name='Oz' | n_mentions=801

cluster_id=111 | canonical_name='the travelers' | n_mentions=598

cluster_id=76 | canonical_name='the Wicked Witch' | n_mentions=357

cluster_id=226 | canonical_name='the Tin Woodman' | n_mentions=258

cluster_id=20 | canonical_name='Toto' | n_mentions=179

cluster_id=60 | canonical_name='the Emerald City' | n_mentions=171

cluster_id=41 | canonical_name='her friends' | n_mentions=140

cluster_id=4 | canonical_name='Aunt Em' | n_mentions=117

cluster_id=252 | canonical_name='the people' | n_mentions=99

cluster_id=67 | canonical_name='the Winkies' | n_mentions=94

cluster_id=1 | 

### Ontology cluster typing

In [186]:
ontology_n_mentions = None  # None = type ALL mentions for each requested cluster
ontology_random_seed = 67

In [187]:
if ONTOLOGY_TYPED_DOC_PATH.exists():
    print(f"[ontology typing] Loading ontology-typed Doc from {ONTOLOGY_TYPED_DOC_PATH}")
    doc = load_doc(ONTOLOGY_TYPED_DOC_PATH)
else:
    if not ONTOLOGY_TTL_PATH.exists():
        raise FileNotFoundError(
            f"Ontology TTL not found: {ONTOLOGY_TTL_PATH}. "
            "Create a networkx.DiGraph by any external method, or place the TTL here."
        )

    jsonl_paths_by_cluster_id = export_ontology_evidence_jsonls(
        doc,
        ontology_graph,
        OntologyEvidenceExportConfig(
            n_mentions_per_cluster=ontology_n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            traversal_config=OntologyTraversalConfig(
                skip_single_root=True,
                include_stay_option=True,
                force_leaf=False,
            ),
            scoring_config=OntologyScoringConfig(
                subject_aware=True,
            ),
            mention_weight_config=OntologyMentionWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64,
            ),

            chunk_size=16,

            overwrite_jsonl=False,
            resume_from_jsonl=True,

            random_seed=ontology_random_seed,
            sort_sample_by_cluster_order=True,
            print_progress=True,
        ),
    )

    print(jsonl_paths_by_cluster_id)

    n_part = "all" if ontology_n_mentions is None else str(ontology_n_mentions)
    ontology_folder = OUTPUT_ROOT / "ontology_typing" / n_part

    doc = annotate_doc_with_ontology_folder(
        doc,
        ontology_graph,
        ontology_folder,
        config=OntologyAnnotationConfig(
            use_mention_weight=True,
        ),
    )
    save_doc(doc, ONTOLOGY_TYPED_DOC_PATH)
    print("\n\n\n[ontology typing] Saved annotated Doc to:", ONTOLOGY_TYPED_DOC_PATH)

ontology_layer = require_ontology_layer(doc)
print(ontology_layer.summary())

[ontology typing] Loading ontology-typed Doc from c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ontology_typed_doc.pkl
{'n_ontology_clusters': 566, 'n_ontology_classes_used': 15}


In [188]:
coref = doc._.coref_layer
ontology_layer = doc._.ontology_layer

rows = []

for cluster_id, cluster in sorted(coref.clusters.items()):
    annotation = ontology_layer.cluster(cluster_id)

    rows.append(
        {
            "cluster_id": cluster_id,
            "canonical_name": cluster.canonical_name,
            "semantic_type": cluster.semantic_type,
            "n_mentions": len(cluster.mention_ids),
            "ontology_class_label": annotation.class_label,
            "ontology_class_human_readable_label": annotation.class_human_readable_label,
        }
    )

ontology_clusters_df = pd.DataFrame(rows)

ontology_clusters_df

,cluster_id,canonical_name,semantic_type,n_mentions,ontology_class_label,ontology_class_human_readable_label
0,0,the great Kansas prairies,UNKNOWN,1,Site,Site
1,1,Kansas,UNKNOWN,93,Region,Region
2,2,Uncle Henry,UNKNOWN,49,Human character,Human Character
3,3,a farmer,UNKNOWN,29,Human character,Human Character
4,4,Aunt Em,UNKNOWN,117,Human character,Human Character
...,...,...,...,...,...,...
561,561,no one,UNKNOWN,1,Narrative entity,Narrative Entity
562,562,the people,UNKNOWN,1,Belief or value,Belief Or Value
563,563,the people,UNKNOWN,1,Site,Site
564,564,her loving comrades,UNKNOWN,1,Belief or value,Belief Or Value


### OCEAN scoring

In [189]:
cluster_ids = [8, 97, 153, 48, 111, 76, 226, 20, 41, 4, 252, 67, 35, 540, 238, 335, 292, 201, 460, 533, 326, 36, 2, 22, 221, 387]
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [161]:
if OCEAN_SCORED_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading ocean_scoring-annotated Doc from {OCEAN_SCORED_DOC_PATH}")
    doc = load_doc(OCEAN_SCORED_DOC_PATH)
else:

    csv_paths_by_cluster_id = export_ocean_probability_csvs(
    doc,
    OceanProbabilityExportConfig(
        cluster_ids=cluster_ids,
        n_mentions_per_cluster=n_mentions,
        output_root="./outputs",

        context_config=ContextConfig(
            n_sentences_before=0,
            n_sentences_after=0,
            mark_mention=True,
            deduplicate=False,
        ),
        rendering_config=MentionRenderingConfig(
            canonicalize_simple_mentions=True,
            keep_first_second_person=True,
        ),
        scoring_config=OceanScoringConfig(
            subject_aware=True,
        ),
        weight_config=OceanWeightConfig(),
        nli_config=DirectNLIConfig(
            pair_batch_size=64, # on stronger hardware can be safely set to 64
        ),

        chunk_size=64,

        overwrite_csv=False,
        resume_from_csv=True,
        use_sqlite_cache=True,

        random_seed=random_seed,
        sort_sample_by_cluster_order=True,
        return_dataframes=False,
        print_progress=True,
    ),
)


    print(csv_paths_by_cluster_id)


    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = Path("./outputs") / "OCEAN_profiles" / n_part

    doc = annotate_doc_with_ocean_folder(
        doc,
        ocean_folder,
    )
    save_doc(doc, OCEAN_SCORED_DOC_PATH)
    print("\n\n\n[OCEAN scoring] Saved annotated Doc to:", OCEAN_SCORED_DOC_PATH)

ocean = require_ocean_layer(doc)
print(ocean.summary())
#### test

[OCEAN scoring] Loading ocean_scoring-annotated Doc from c:\Users\Bobby\Desktop\real shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ocean_scored_doc.pkl
{'n_ocean_mentions': 7274, 'n_ocean_clusters': 26}


In [162]:
coref = doc._.coref_layer
ocean = doc._.ocean_layer

for cluster_id in cluster_ids:
    print(coref.clusters[cluster_id].canonical_name)
    print(ocean.cluster(cluster_id).scores.as_dict())

Dorothy
{'openness': 55.28, 'conscientiousness': 51.26, 'extraversion': 53.0, 'agreeableness': 67.85, 'neuroticism': 42.27}
the Scarecrow
{'openness': 46.12, 'conscientiousness': 45.43, 'extraversion': 44.12, 'agreeableness': 54.87, 'neuroticism': 48.08}
the Lion
{'openness': 45.49, 'conscientiousness': 48.0, 'extraversion': 40.77, 'agreeableness': 48.42, 'neuroticism': 52.92}
Oz
{'openness': 47.44, 'conscientiousness': 51.73, 'extraversion': 48.53, 'agreeableness': 56.74, 'neuroticism': 39.64}
the travelers
{'openness': 57.08, 'conscientiousness': 53.5, 'extraversion': 53.54, 'agreeableness': 68.59, 'neuroticism': 34.87}
the Wicked Witch
{'openness': 45.25, 'conscientiousness': 52.17, 'extraversion': 47.27, 'agreeableness': 23.08, 'neuroticism': 43.77}
the Tin Woodman
{'openness': 44.3, 'conscientiousness': 52.85, 'extraversion': 52.64, 'agreeableness': 61.54, 'neuroticism': 35.8}
Toto
{'openness': 59.94, 'conscientiousness': 50.53, 'extraversion': 62.84, 'agreeableness': 64.76, 'neur

## Edge extraction

In [163]:
SUBJECT_DEPS = {
    "nsubj",
    "nsubjpass",
    "csubj",
    "csubjpass",
}

PASSIVE_SUBJECT_DEPS = {
    "nsubjpass",
    "csubjpass",
}

OBJECT_DEPS = {
    "dobj",      # spaCy English models often use this
    "obj",       # Universal Dependencies-style label
    "iobj",
    "dative",
    "attr",      # copular / attributive complement
    "oprd",
}

PREP_DEPS = {
    "prep",
}

PREP_OBJECT_DEPS = {
    "pobj",
    "pcomp",
}

AUX_PASSIVE_DEPS = {
    "auxpass",
}

NEGATION_DEPS = {
    "neg",
}

PARTICLE_DEPS = {
    "prt",
}


@dataclass(frozen=True)
class RelationCandidate:
    source_cluster_id: int
    source_canonical_name: str
    source_cluster_type: str

    predicate: str
    predicate_surface: str
    predicate_token_i: int

    target_cluster_id: int
    target_canonical_name: str
    target_cluster_type: str

    sentence_start: int
    sentence_end: int
    sentence_text: str

    source_mention_id: int | None
    target_mention_id: int | None

    source_token_i: int
    target_token_i: int

    object_dependency: str
    preposition: str | None

    is_passive: bool
    is_negated: bool


def _ensure_dependency_parse(doc) -> None:
    if not doc.has_annotation("DEP"):
        raise ValueError(
            "This Doc has no dependency parse. "
            "You need a spaCy pipeline with the parser enabled."
        )

    if not doc.has_annotation("POS"):
        raise ValueError(
            "This Doc has no POS annotation. "
            "You need POS tags to reliably identify verbal predicates."
        )


def _token_mentions(coref, token) -> list:
    """
    Resolve a syntactic token to coreference mentions.

    Prefer exact syntactic-head match when available.
    Fall back to mentions covering the token.
    """
    head_mentions = coref.mentions_from_head_token(token.i)
    if head_mentions:
        return head_mentions

    return coref.mentions_from_token(token.i)


def _cluster_records_for_token(coref, token) -> list[tuple[int, int]]:
    """
    Return pairs:
        (mention_id, cluster_id)

    Multiple results are possible when mentions overlap.
    Do not silently collapse them here.
    """
    pairs = []

    for mention in _token_mentions(coref, token):
        if mention.cluster_id not in coref.clusters:
            continue
        pairs.append((mention.mention_id, mention.cluster_id))

    # Deduplicate while preserving deterministic order
    seen = set()
    unique_pairs = []
    for pair in pairs:
        if pair in seen:
            continue
        unique_pairs.append(pair)
        seen.add(pair)

    return unique_pairs


def _is_verbal_predicate(token) -> bool:
    return (
        token.pos_ == "VERB"
        or (token.pos_ == "AUX" and token.dep_ in {"ROOT", "cop"})
    )


def _is_passive(predicate) -> bool:
    if any(child.dep_ in AUX_PASSIVE_DEPS for child in predicate.children):
        return True

    if any(child.dep_ in PASSIVE_SUBJECT_DEPS for child in predicate.children):
        return True

    return False


def _is_negated(predicate) -> bool:
    return any(child.dep_ in NEGATION_DEPS for child in predicate.children)


def _predicate_label(predicate, *, preposition=None) -> str:
    """
    Build a normalized predicate label.

    Examples:
        give
        look_at
        go_to
        pick_up
    """
    base = (predicate.lemma_ or predicate.text).lower()

    particles = [
        child.text.lower()
        for child in sorted(predicate.children, key=lambda t: t.i)
        if child.dep_ in PARTICLE_DEPS
    ]

    parts = [base, *particles]

    if preposition is not None:
        parts.append((preposition.lemma_ or preposition.text).lower())

    return "_".join(parts)


def _subjects_of(predicate) -> list:
    return [
        child
        for child in predicate.children
        if child.dep_ in SUBJECT_DEPS
    ]


def _direct_objects_of(predicate) -> list[tuple[object, str, object | None]]:
    """
    Return triples:
        (object_token, object_dependency, preposition_token_or_None)
    """
    objects = []

    for child in predicate.children:
        if child.dep_ in OBJECT_DEPS:
            objects.append((child, child.dep_, None))

    return objects


def _prepositional_objects_of(predicate) -> list[tuple[object, str, object]]:
    """
    Extract cases like:
        went to Camelot
        spoke with Arthur
        looked at Merlin

    The edge predicate becomes:
        go_to
        speak_with
        look_at
    """
    objects = []

    for prep in predicate.children:
        if prep.dep_ not in PREP_DEPS:
            continue

        for pobj in prep.children:
            if pobj.dep_ in PREP_OBJECT_DEPS:
                objects.append((pobj, pobj.dep_, prep))

    return objects


def extract_relation_candidates(doc) -> pd.DataFrame:
    _ensure_dependency_parse(doc)

    coref = require_coref_layer(doc)
    ontology_layer = require_ontology_layer(doc)

    rows: list[RelationCandidate] = []

    for sent in doc.sents:
        for predicate in sent:
            if not _is_verbal_predicate(predicate):
                continue

            subjects = _subjects_of(predicate)
            objects = [
                *_direct_objects_of(predicate),
                *_prepositional_objects_of(predicate),
            ]

            if not subjects or not objects:
                continue

            passive = _is_passive(predicate)
            negated = _is_negated(predicate)

            for subject_token in subjects:
                source_pairs = _cluster_records_for_token(coref, subject_token)

                if not source_pairs:
                    continue

                for object_token, object_dep, prep_token in objects:
                    target_pairs = _cluster_records_for_token(coref, object_token)

                    if not target_pairs:
                        continue

                    predicate_label = _predicate_label(
                        predicate,
                        preposition=prep_token,
                    )

                    for (source_mention_id, source_cluster_id), (
                        target_mention_id,
                        target_cluster_id,
                    ) in product(source_pairs, target_pairs):

                        if source_cluster_id == target_cluster_id:
                            continue

                        source_cluster = coref.clusters[source_cluster_id]
                        target_cluster = coref.clusters[target_cluster_id]

                        source_cluster_type = (
                            ontology_layer
                            .cluster(source_cluster_id)
                            .class_label
                        )

                        target_cluster_type = (
                            ontology_layer
                            .cluster(target_cluster_id)
                            .class_label
                        )

                        rows.append(
                            RelationCandidate(
                                source_cluster_id=source_cluster_id,
                                source_canonical_name=source_cluster.canonical_name,
                                source_cluster_type=source_cluster_type,

                                predicate=predicate_label,
                                predicate_surface=predicate.text,
                                predicate_token_i=predicate.i,

                                target_cluster_id=target_cluster_id,
                                target_canonical_name=target_cluster.canonical_name,
                                target_cluster_type=target_cluster_type,

                                sentence_start=sent.start,
                                sentence_end=sent.end,
                                sentence_text=sent.text,

                                source_mention_id=source_mention_id,
                                target_mention_id=target_mention_id,

                                source_token_i=subject_token.i,
                                target_token_i=object_token.i,

                                object_dependency=object_dep,
                                preposition=prep_token.text if prep_token is not None else None,

                                is_passive=passive,
                                is_negated=negated,
                            )
                        )

    return pd.DataFrame(
        [asdict(row) for row in rows],
        columns=list(RelationCandidate.__dataclass_fields__),
    )

In [164]:
relation_candidates_df = extract_relation_candidates(doc)

relation_candidates_df[
    [
        "source_cluster_id",
        "source_canonical_name",
        "source_cluster_type",
        "predicate",
        "target_cluster_id",
        "target_canonical_name",
        "target_cluster_type",
        "sentence_text",
        "is_passive",
        "is_negated",
    ]
]

,source_cluster_id,source_canonical_name,source_cluster_type,predicate,target_cluster_id,target_canonical_name,target_cluster_type,sentence_text,is_passive,is_negated
0,4,Aunt Em,Human character,look_at,8,Dorothy,Human character,"When Dorothy, who was an orphan, first came to...",False,False
1,8,Dorothy,Human character,play_with,20,Toto,Non-human character,"Toto played all day long, and Dorothy played w...",False,False
2,2,Uncle Henry,Human character,call_to,4,Aunt Em,Human character,"""There's a cyclone coming, Em,"" he called to h...",False,False
3,8,Dorothy,Human character,catch,20,Toto,Non-human character,Dorothy caught Toto at last and started to fol...,False,False
4,8,Dorothy,Human character,rock_like,23,a baby in a cradle,Belief or value,"After the first few whirls around, and one oth...",True,False
...,...,...,...,...,...,...,...,...,...,...
1350,255,a farmhouse,Site,carry,558,the old one,Place,For she was sitting on the broad Kansas prairi...,False,False
1351,255,a farmhouse,Site,carry,565,the old one,Place,For she was sitting on the broad Kansas prairi...,False,False
1352,2,Uncle Henry,Human character,milk_in,559,the barnyard,Site,Uncle Henry was milking the cows in the barnya...,False,False
1353,8,Dorothy,Human character,run_toward,540,Glinda,Non-human character,Aunt Em had just come out of the house to wate...,False,False
